In [ ]:
import yfinance as yf
import pandas as pd

TICKERS = {
    "Delta Air Lines": "DAL",
    "Air Canada": "AC.TO",
    "Lufthansa": "LHA.DE",
    "Turkish Airlines": "THYAO.IS",
    "Finnair": "FIA1S.HE",
    "Air France": "AF.PA",
    "KLM Royal Dutch Airlines": "AF.PA",
    "British Airways": "IAG.L",
    "Iberia": "IAG.L",
    "Vueling": "IAG.L",
    "Aegean Airlines": "AEGN.AT",
    "Qantas Airways": "QAN.AX",
    "Air New Zealand": "AIR.NZ",
    "ANA All Nippon Airways": "9202.T",
    "Japan Airlines": "9201.T",
    "Korean Air": "003490.KS",
    "Singapore Airlines": "C6L.SI",
    "Cathay Pacific Airways": "0293.HK",
    "China Southern Airlines": "1055.HK",
    "AirAsia": "5099.KL",
    "IndiGo": "INDIGO.NS",
    "China Airlines": "2610.TW",
    "EVA Air": "2618.TW",
    "STARLUX Airlines": "2646.TW",
    "LATAM Airlines": "LTM.SN",
    "Garuda Indonesia": "GIAA.JK",
    "Air Astana": "AIRA.IL",
}

# 1) unikal ticker list
tickers = sorted(set(TICKERS.values()))

# 2) OHLCV + Adj Close (hamısını bir dəfəyə çəkir)
px = yf.download(tickers, period="5y", interval="1d", group_by="ticker", auto_adjust=False, threads=True)

# 3) Long format
out = []
for airline, tkr in TICKERS.items():
    if tkr not in px.columns.get_level_values(0):
        continue
    df = px[tkr].copy()
    df["airline_name"] = airline
    df["ticker"] = tkr
    df = df.reset_index().rename(columns={"Date":"date", "Adj Close":"adj_close", "Open":"open", "High":"high", "Low":"low", "Close":"close", "Volume":"volume"})
    out.append(df[["date","airline_name","ticker","adj_close","close","high","low","open","volume"]])

stock_data_df = pd.concat(out, ignore_index=True)
stock_data_df.to_csv("stock_data_from_yahoo.csv", index=False)

[*********************100%***********************]  24 of 24 completed


Saved: (35289, 9)


In [ ]:
import yfinance as yf
import pandas as pd


TICKERS = {

    "Delta Air Lines": "DAL",
    "Air Canada": "AC.TO",
    "Lufthansa": "LHA.DE",
    "Turkish Airlines": "THYAO.IS",
    "Finnair": "FIA1S.HE",
    "Air France": "AF.PA",
    "KLM Royal Dutch Airlines": "AF.PA",
    "British Airways": "IAG.L",
    "Iberia": "IAG.L",
    "Vueling": "IAG.L",
    "Aegean Airlines": "AEGN.AT",
    "Qantas Airways": "QAN.AX",
    "Air New Zealand": "AIR.NZ",
    "ANA All Nippon Airways": "9202.T",
    "Japan Airlines": "9201.T",
    "Korean Air": "003490.KS",
    "Singapore Airlines": "C6L.SI",
    "Cathay Pacific Airways": "0293.HK",
    "China Southern Airlines": "1055.HK",
    "AirAsia": "5099.KL",
    "IndiGo": "INDIGO.NS",
    "China Airlines": "2610.TW",
    "EVA Air": "2618.TW",
    "STARLUX Airlines": "2646.TW",
    "LATAM Airlines": "LTM.SN",
    "Garuda Indonesia": "GIAA.JK",
    "Air Astana": "AIRA.IL",
}


def safe_get(d, key):
    try:
        v = d.get(key, None)
        return None if v in ("None", "", "nan") else v
    except Exception:
        return None

def pick_row(df: pd.DataFrame, row_name: str):
    if df is None or df.empty:
        return None
    # yfinance statement index adları bəzən fərqlənir; exact match + fallback
    if row_name in df.index:
        return df.loc[row_name]
    # fallback: case-insensitive contains
    low = row_name.lower()
    for idx in df.index:
        if low == str(idx).lower():
            return df.loc[idx]
    for idx in df.index:
        if low in str(idx).lower():
            return df.loc[idx]
    return None

def to_long_series(series: pd.Series, metric: str):
    # series index: columns are dates/periods
    if series is None or len(series) == 0:
        return pd.DataFrame(columns=["period_end", "metric", "value"])
    s = series.copy()
    s.index = pd.to_datetime(s.index, errors="coerce")
    out = pd.DataFrame({"period_end": s.index, "metric": metric, "value": s.values})
    out = out.dropna(subset=["period_end"])
    return out

fund_rows = []
statement_rows = []
missing = []

for airline, ticker in TICKERS.items():
    try:
        t = yf.Ticker(ticker)

        # 1) Company info / ratios / market cap
        info = t.get_info()  # yfinance>=0.2.x
        fund_rows.append({
            "airline_name": airline,
            "ticker": ticker,
            "currency": safe_get(info, "currency"),
            "exchange": safe_get(info, "exchange"),
            "market_cap": safe_get(info, "marketCap"),
            "enterprise_value": safe_get(info, "enterpriseValue"),
            "trailing_pe": safe_get(info, "trailingPE"),
            "forward_pe": safe_get(info, "forwardPE"),
            "price_to_book": safe_get(info, "priceToBook"),
            "profit_margins": safe_get(info, "profitMargins"),
            "operating_margins": safe_get(info, "operatingMargins"),
            "return_on_assets": safe_get(info, "returnOnAssets"),
            "return_on_equity": safe_get(info, "returnOnEquity"),
            "debt_to_equity": safe_get(info, "debtToEquity"),
            "total_cash": safe_get(info, "totalCash"),
            "total_debt": safe_get(info, "totalDebt"),
            "revenue_ttm": safe_get(info, "totalRevenue"),
            "ebitda_ttm": safe_get(info, "ebitda"),
        })

        # 2) Income statement (Annual)
        # yfinance-də statement adları: income_stmt (yeni), financials (köhnə)
        income = getattr(t, "income_stmt", None)
        if income is None or income is False:
            income = t.financials  # fallback
        # income columns: periods; index: metrics

        total_rev = pick_row(income, "Total Revenue")
        op_income = pick_row(income, "Operating Income")
        net_income = pick_row(income, "Net Income")
        ebitda = pick_row(income, "EBITDA")
        cost_rev = pick_row(income, "Cost Of Revenue")  # proxy üçün
        op_exp = pick_row(income, "Operating Expense")  # proxy üçün (bəzən yoxdur)

        for metric_name, series in [
            ("total_revenue", total_rev),
            ("operating_income", op_income),
            ("net_income", net_income),
            ("ebitda", ebitda),
            ("cost_of_revenue", cost_rev),
            ("operating_expense", op_exp),
        ]:
            long_df = to_long_series(series, metric_name)
            if not long_df.empty:
                long_df["airline_name"] = airline
                long_df["ticker"] = ticker
                statement_rows.append(long_df)

    except Exception as e:
        missing.append({"airline_name": airline, "ticker": ticker, "reason": str(e)})

# Save fundamentals
fund_df = pd.DataFrame(fund_rows)
fund_df.to_csv("airline_fundamentals.csv", index=False)

# Save income statement metrics (long format)
stmt_df = pd.concat(statement_rows, ignore_index=True) if statement_rows else pd.DataFrame()
stmt_df = stmt_df[["airline_name", "ticker", "period_end", "metric", "value"]]
stmt_df.to_csv("airline_income_statement_metrics.csv", index=False)

# Save missing
pd.DataFrame(missing).to_csv("missing_or_failed.csv", index=False)

# (İSTƏSƏN) 3) Daily prices (son 5 il)
tickers_unique = sorted(set(TICKERS.values()))
px = yf.download(tickers_unique, period="5y", interval="1d", group_by="ticker", auto_adjust=False, threads=True)

out = []
for airline, tkr in TICKERS.items():
    if tkr not in px.columns.get_level_values(0):
        continue
    df = px[tkr].copy().reset_index()
    df["airline_name"] = airline
    df["ticker"] = tkr
    df = df.rename(columns={"Date":"date", "Adj Close":"adj_close", "Open":"open", "High":"high", "Low":"low", "Close":"close", "Volume":"volume"})
    out.append(df[["date","airline_name","ticker","adj_close","close","high","low","open","volume"]])

prices_df = pd.concat(out, ignore_index=True) if out else pd.DataFrame()
prices_df.to_csv("airline_prices_5y.csv", index=False)

✅ Saved files:
- airline_fundamentals.csv
- airline_income_statement_metrics.csv
- missing_or_failed.csv
Rows fundamentals: 27 | Rows statement: 696 | Missing: 0


[*********************100%***********************]  24 of 24 completed


- airline_prices_5y.csv | Rows prices: 35289


In [7]:
import pandas as pd
df = pd.read_csv("airline_income_statement_metrics.csv")
df

,airline_name,ticker,period_end,metric,value
0,Delta Air Lines,DAL,2025-12-31,total_revenue,6.336400e+10
1,Delta Air Lines,DAL,2024-12-31,total_revenue,6.164300e+10
2,Delta Air Lines,DAL,2023-12-31,total_revenue,5.804800e+10
3,Delta Air Lines,DAL,2022-12-31,total_revenue,5.058200e+10
4,Delta Air Lines,DAL,2021-12-31,total_revenue,NaN
...,...,...,...,...,...
691,Air Astana,AIRA.IL,2021-12-31,cost_of_revenue,5.947310e+08
692,Air Astana,AIRA.IL,2024-12-31,operating_expense,1.112950e+08
693,Air Astana,AIRA.IL,2023-12-31,operating_expense,1.004270e+08
694,Air Astana,AIRA.IL,2022-12-31,operating_expense,8.293400e+07


In [10]:
df["airline_name"].unique()

array(['Delta Air Lines', 'Air Canada', 'Lufthansa', 'Turkish Airlines',
       'Finnair', 'Air France', 'KLM Royal Dutch Airlines',
       'British Airways', 'Iberia', 'Vueling', 'Aegean Airlines',
       'Qantas Airways', 'Air New Zealand', 'ANA All Nippon Airways',
       'Japan Airlines', 'Korean Air', 'Singapore Airlines',
       'Cathay Pacific Airways', 'China Southern Airlines', 'AirAsia',
       'IndiGo', 'China Airlines', 'EVA Air', 'STARLUX Airlines',
       'LATAM Airlines', 'Garuda Indonesia', 'Air Astana'], dtype=object)